# Tutorial 1 — Basic Financial Models

Discrete-time stochastic simulators for three everyday business processes: a **stock price** reacting to a market event, a **resource price** hit by a supply disruption, and **product demand** lifted by a marketing campaign.

Each model is a one-line geometric random walk $x_{t+1} = x_t(1 + \varepsilon_t)$ with $\varepsilon_t \sim \mathcal{N}(\mu, \sigma)$, into which we inject a single deterministic shock. That makes the effect of the shock analytically exact and easy to validate.

## Setup

Lock the seed and import the three simulators.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sim_lab.core import (
    StockMarketSimulation,
    ResourceFluctuationsSimulation,
    ProductPopularitySimulation,
)

random_seed = 42
np.random.seed(random_seed)
print(f"Reproducibility seed locked: {random_seed}")

## 1. Stock Market — event impact

On `event_day` the random step is *replaced* by a deterministic move, so the price that day is exactly

$$p_{\text{event}} = p_{\text{event}-1}\,(1 + \text{event\_impact}).$$

In [ ]:
sm = StockMarketSimulation(
    start_price=100.0, days=60, volatility=0.012, drift=0.0005,
    event_day=20, event_impact=0.10, random_seed=random_seed,
)
prices = sm.run_simulation()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prices, label='price')
ax.axvline(20, color='crimson', ls='--', alpha=0.7, label='event day')
ax.set_xlabel('day'); ax.set_ylabel('price')
ax.set_title('Stock price with a +10% market event on day 20')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# Analytic check: the event-day move is exact, not noisy.
expected = prices[19] * (1 + 0.10)
assert np.isclose(prices[20], expected), 'event-day price mismatch'
print(f'price[19] = {prices[19]:.4f}')
print(f'price[20] = {prices[20]:.4f}  (= prior * 1.10 = {expected:.4f})')

## 2. Resource Fluctuations — supply disruption

A disruption acts like the stock event but on a commodity: on `supply_disruption_day` the price jumps by `disruption_severity`, after which the normal drift/volatility resume and the spike **decays** away.

In [ ]:
rf = ResourceFluctuationsSimulation(
    start_price=50.0, days=60, volatility=0.012, drift=0.0,
    supply_disruption_day=20, disruption_severity=0.30, random_seed=random_seed,
)
rprices = rf.run_simulation()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rprices, label='resource price')
ax.axvline(20, color='darkorange', ls='--', alpha=0.7, label='disruption')
ax.set_xlabel('day'); ax.set_ylabel('price')
ax.set_title('Resource price with a +30% supply disruption on day 20')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

spike = rprices[20] / rprices[19]
after = rprices[20] / rprices[-1]   # how much of the spike survives to the end
assert np.isclose(spike, 1.30), 'disruption spike mismatch'
print(f'spike on disruption day = {spike:.3f}  (expected 1.30)')
print(f'spike retained at end   = {after:.3f}  -> decays toward drift')

## 3. Product Popularity — marketing campaign

Demand grows from a natural growth rate plus daily marketing impact. A `promotion_day` multiplies that day's demand by $(1 + \text{promotion\_effectiveness})$, permanently lifting the whole subsequent trajectory. We compare a run **with** vs **without** the campaign.

In [ ]:
common = dict(start_demand=100.0, days=60, growth_rate=0.01, marketing_impact=0.02,
             random_seed=random_seed)
no_camp = ProductPopularitySimulation(promotion_day=None, promotion_effectiveness=0.0, **common)
camp    = ProductPopularitySimulation(promotion_day=30, promotion_effectiveness=0.50, **common)
d_no = no_camp.run_simulation()
d_yes = camp.run_simulation()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(d_no, label='no campaign')
ax.plot(d_yes, label='campaign on day 30')
ax.axvline(30, color='green', ls='--', alpha=0.7, label='campaign')
ax.set_xlabel('day'); ax.set_ylabel('demand')
ax.set_title('Product demand with and without a marketing campaign')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

cum_no, cum_yes = sum(d_no), sum(d_yes)
assert cum_yes > cum_no, 'campaign should raise cumulative demand'
print(f'cumulative demand  no campaign = {cum_no:,.0f}')
print(f'cumulative demand  w/ campaign  = {cum_yes:,.0f}  (+{100*(cum_yes-cum_no)/cum_no:.1f}%)')

## Validation & interpretation

| Model | Law | Check |
|---|---|---|
| Stock Market | event-day price = prior × (1 + impact) | `prices[20] == prices[19]*1.10` ✅ |
| Resource Fluct. | disruption spike = 1 + severity, then decays | spike `1.30`, retained `0.30` ✅ |
| Product Pop. | a one-day promotion raises *cumulative* demand | `22079 > 16305` ✅ |

All three simulators share the same geometric-walk skeleton, so each shock has an **exact** effect on its day — the validation is an equality, not a statistic. The disruption spike decays because, after the shock day, only the drift/volatility term acts and (with zero drift) the expected path is flat. The campaign's effect compounds: because demand is multiplicative, a single 50% lift on day 30 raises *every* later value, which is why the cumulative gain (~35%) far exceeds the one-day 50% bump.